# 🎭 Live Emotion Detection
**Using your trained facialemotionmodel**

### Steps:
1. Run Cell 1 — Install & import
2. Run Cell 2 — Upload your model files
3. Run Cell 3 — Load model
4. Run Cell 4 — Start live camera

In [3]:
# ── Cell 1: Install & Imports ──────────────────────────────────
!pip install -q opencv-python-headless

import cv2
import numpy as np
from keras.models import model_from_json
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode
import base64

print('✅ All imports done')


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: C:\Users\NILAVRA GHOSH\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


ModuleNotFoundError: No module named 'cv2'

In [ ]:
# ── Cell 2: Upload your model files ────────────────────────────
from google.colab import files

print('📁 Upload facialemotionmodel.json and facialemotionmodel.h5')
uploaded = files.upload()
print('✅ Uploaded:', list(uploaded.keys()))

In [ ]:
# ── Cell 3: Load your trained model ────────────────────────────
import tensorflow as tf

# Load model architecture
json_file = open('facialemotionmodel.json', 'r')
model_json = json_file.read()
json_file.close()

model = model_from_json(model_json, custom_objects={'Sequential': tf.keras.Sequential})
model.load_weights('facialemotionmodel.h5')

print('✅ Model loaded successfully!')
model.summary()

# Emotion labels — must match training order
LABELS = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']
EMOJI  = {'angry':'😠', 'disgust':'🤢', 'fear':'😨', 'happy':'😊',
           'neutral':'😐', 'sad':'😢', 'surprise':'😲'}

# Load Haar cascade face detector
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
print('✅ Face detector ready!')

In [ ]:
# ── Cell 4: Live Camera Emotion Detection ──────────────────────
# Press Escape or click Stop (■) in Colab to end the stream

def preprocess_face(gray_face):
    """Preprocess face exactly like training pipeline"""
    face = cv2.resize(gray_face, (48, 48))
    face = face.astype('float32') / 255.0
    face = np.reshape(face, (1, 48, 48, 1))
    return face

def draw_overlay(frame, x, y, w, h, emotion, confidence):
    """Draw styled bounding box and label"""
    # Green bounding box
    cv2.rectangle(frame, (x, y), (x+w, y+h), (74, 124, 89), 2)
    
    # Background for label
    label_text = f'{emotion.upper()}  {confidence}%'
    (tw, th), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
    cv2.rectangle(frame, (x, y-35), (x+tw+10, y), (74, 124, 89), -1)
    cv2.putText(frame, label_text, (x+5, y-10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    return frame

# JavaScript for camera stream
display(Javascript('''
    window._mbStop = false;
    window._mbResolve = null;
    window._mbInited = false;

    async function mbInit() {
        if (window._mbInited) return;
        window._mbInited = true;

        // Container
        const wrap = document.createElement('div');
        wrap.style.cssText = 'max-width:660px;margin:10px 0;font-family:sans-serif';
        document.body.appendChild(wrap);

        // Title
        const title = document.createElement('div');
        title.style.cssText = 'background:#2d5a3d;color:#fff;padding:8px 14px;border-radius:8px 8px 0 0;font-size:15px;font-weight:600';
        title.innerHTML = '🎭 MindBloom — Live Emotion Detection';
        wrap.appendChild(title);

        // Status bar
        window._mbStatus = document.createElement('div');
        window._mbStatus.style.cssText = 'background:#162019;color:#7aab8a;padding:6px 14px;font-size:13px;min-height:28px';
        window._mbStatus.innerText = '🔍 Starting camera...';
        wrap.appendChild(window._mbStatus);

        // Video
        window._mbVideo = document.createElement('video');
        window._mbVideo.style.cssText = 'width:100%;display:block;transform:scaleX(-1);background:#000';
        window._mbVideo.autoplay = true;
        wrap.appendChild(window._mbVideo);

        // Processed frame display
        window._mbImg = document.createElement('img');
        window._mbImg.style.cssText = 'width:100%;display:none;';
        wrap.appendChild(window._mbImg);

        // Footer
        const footer = document.createElement('div');
        footer.style.cssText = 'background:#0a1309;color:#5a7a62;padding:6px 14px;border-radius:0 0 8px 8px;font-size:11px';
        footer.innerText = 'Press Escape or Colab Stop button (■) to end session';
        wrap.appendChild(footer);

        // Canvas for frame capture
        window._mbCanvas = document.createElement('canvas');
        window._mbCanvas.width  = 640;
        window._mbCanvas.height = 480;

        // Start webcam
        try {
            const stream = await navigator.mediaDevices.getUserMedia({
                video: { width:640, height:480, facingMode:'user', frameRate:{ideal:30} }
            });
            window._mbVideo.srcObject = stream;
            await window._mbVideo.play();
            window._mbStatus.innerText = '✅ Camera ready — detecting emotions...';
        } catch(e) {
            window._mbStatus.innerText = '❌ Camera error: ' + e.message;
            window._mbStop = true;
        }

        // Escape key to stop
        document.addEventListener('keydown', e => {
            if (e.key === 'Escape') { window._mbStop = true; }
        });

        // Frame loop
        function loop() {
            if (!window._mbStop) requestAnimationFrame(loop);
            if (window._mbResolve) {
                if (window._mbStop) {
                    window._mbResolve('');
                } else {
                    window._mbCanvas.getContext('2d')
                        .drawImage(window._mbVideo, 0, 0, 640, 480);
                    window._mbResolve(
                        window._mbCanvas.toDataURL('image/jpeg', 0.65));
                }
                window._mbResolve = null;
            }
        }
        requestAnimationFrame(loop);
    }

    async function mbGetFrame(statusText, processedFrame) {
        await mbInit();
        if (statusText)  window._mbStatus.innerText = statusText;
        if (processedFrame) {
            window._mbVideo.style.display = 'none';
            window._mbImg.style.display   = 'block';
            window._mbImg.src = processedFrame;
        }
        if (window._mbStop) return '';
        return new Promise(r => { window._mbResolve = r; });
    }
'''))

print('🎥 Camera UI loaded. Starting detection loop...')
print('⏹  Press Escape or Colab Stop button to stop')

# ── Detection loop ──────────────────────────────────────────────
PREDICT_EVERY = 3      # predict every N frames (lower = more accurate, higher = faster)
frame_count   = 0
last_label    = '🔍 Detecting...'
processed_b64 = ''

while True:
    # Get raw frame from browser
    js_reply = eval_js(f'mbGetFrame("{last_label}", "{processed_b64}")')
    if not js_reply:
        print('⛔ Stream ended.')
        break

    # Decode JPEG frame
    img_bytes = b64decode(js_reply.split(',')[1])
    arr       = np.frombuffer(img_bytes, dtype=np.uint8)
    frame     = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if frame is None:
        continue

    frame_count += 1

    # Mirror frame (matches video mirror transform)
    frame = cv2.flip(frame, 1)

    if frame_count % PREDICT_EVERY == 0:
        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Detect faces
        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=1.1,
            minNeighbors=5,
            minSize=(30, 30)
        )

        if len(faces) > 0:
            # Use the largest detected face
            faces_sorted = sorted(faces, key=lambda f: f[2]*f[3], reverse=True)
            (x, y, w, h) = faces_sorted[0]

            # Preprocess exactly like training
            face_roi   = gray[y:y+h, x:x+w]
            face_input = preprocess_face(face_roi)

            # Predict
            preds      = model.predict(face_input, verbose=0)[0]
            idx        = preds.argmax()
            emotion    = LABELS[idx]
            confidence = int(preds[idx] * 100)
            emoji      = EMOJI[emotion]

            # Draw overlay on frame
            frame = draw_overlay(frame, x, y, w, h, emotion, confidence)
            last_label = f'{emoji} {emotion.upper()}  —  {confidence}% confidence'
        else:
            last_label = '👤 No face detected — look at camera'

    # Encode processed frame and send back to browser
    _, buf      = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 70])
    processed_b64 = 'data:image/jpeg;base64,' + base64.b64encode(buf).decode()

print('✅ Done!')